## <font color = #0288d1>循环神经网络（Rerrent Neural Network, RNN）</font>

> ### 循环神经网络非常适合序列问题，因为它们考虑了数据的基本结构，即数据点的顺序。数据点是一个接一个的输入循神经网络的，修改RNN的隐藏状态，最终输出表示了完整序列。也就是说，<font color="purple">RNN可以处理序列</font>，或者时序数据。而所谓的隐藏的状态只是一个向量。向量的大小取决于你的需要。我们需要指定隐藏维度的数量，也就是指定表示隐藏状态的向量的大小。

## <font color = #0288d1>基本结构</font>

![](../pictures/7-30.png)

### 1. 有一个初始的<font color="purple">**隐藏状态h~i~表示空序列的状态**</font>，通常用0初始化。RNN单元需要两个输入，一个隐藏的状态代表到目前为止的序列状态，另一个是从输入序列按顺序依次输入的数据点。

### 2. 图中假设输入序列中有2个数据点。初始隐藏状态h0和输入序列的第一个数据x0这两个输入被用来生成一个新的隐藏状态（第一个数据点的h0），表示现在的更新状态。

### 3. 然后将输入数据中的第二个数据点x1和上一个的隐藏状态h0作为下一次的输入。新的隐藏状态既是当前步骤的输出，也是下一个步骤的输入之一。

### 4. 如果序列中还有另一个数据点，就一直执行此循环，直到处理完最后一个数据。最后一个隐藏的状态（在图中是h2）也是整个RNN的最终隐藏状态hf。

> ### 特别要注意的是，在多层神经网络中，如前面的CNN，每一层都有自己参数，自己的卷积核等。但是对于RNN来说，<font color="purple">**其实总体看只有一个层，数据被反复的输入到了这个层**</font>。在RNN中，h和x的输入经过变换后相加，最后由tanh函数激活后输出：

### $$ t_{h} = w_{hh}*h_{t - 1} + b_{hh} $$
### $$ t_{x} = w_{ih}*x_{t - 1} + b_{ih} $$
### $$ h_{t} = tanh(t_{h} + t_{x}) $$

> ### 此处需要注意的是，由于在最后一步中th和tx需要相加，因此<font color="purple">**需要这两个向量的维度是一样的才能相加**</font>。
> ### 注意道激活函数没有选用我们喜欢的ReLU函数，而是选择了tanh，主要是tanh能保证变换后的结果在-1和1之间。这使得下一次的数据输入更加可控，当然选择别的激活函数也是可以的。


## <font color = #0288d1>Pytorch的RNN层</font>

### PyTorch提供了对应循环神经网络的nn.RNN层。RNN层负责处理隐藏状态，无论输入序列有多长，这也是实际计算模型中使用的层。我们已经了解RNN的内部工作原理，但Pytorch的RNN提供了更多的选项（例如堆叠和/或双向层），以及关于输入和输出形状的一件棘手的事情。

In [1]:
import torch
import torch.nn as nn
rnn = nn.RNN(input_size=3, hidden_size=5)
rnn.state_dict()

OrderedDict([('weight_ih_l0',
              tensor([[-0.3275,  0.2474, -0.2238],
                      [ 0.0903, -0.3807,  0.2045],
                      [-0.1770,  0.3142, -0.0275],
                      [-0.1323,  0.2867,  0.2767],
                      [-0.3247, -0.1240, -0.0151]])),
             ('weight_hh_l0',
              tensor([[ 0.2413,  0.0928, -0.0847,  0.4039,  0.3302],
                      [-0.1265,  0.0493, -0.3776, -0.3930, -0.2540],
                      [-0.3211,  0.3763,  0.1091,  0.2512,  0.2218],
                      [ 0.0159, -0.2865,  0.3235, -0.1877, -0.1330],
                      [-0.3390,  0.3142,  0.3679,  0.1685, -0.4068]])),
             ('bias_ih_l0',
              tensor([-0.2900, -0.4353,  0.2692,  0.1926,  0.2808])),
             ('bias_hh_l0',
              tensor([-0.1608,  0.2271,  0.1385,  0.0201, -0.3057]))])

### 和前面创建层的代码样式一致，只是参数不同。
+ ### 首先是input参数，该参数指的是序列数据中，每一个数据的特征数。如数据是一系列空间的坐标点，由于每个坐标点含有3个分量，此时input_size=3。
+ ### 第二个参数是隐藏状态的维度，可以根据需要指定任意的维度。需要注意的是这个维度也是单元输出的维度。
+ ### 例如输入数据是空间的点，维度是3，指定隐藏状态的维度是5，根据前面给出的公式，数据序列的wih矩阵为5\*3，隐藏状态的whh矩阵为5\*5，运算后th和tx都是5\*1的向量。最后的结果ht也是5\*1的向量。

+ ### 此外还有一个需要注意的参数：batch_first。batch_first决定输入数据的次序（但不包括隐藏变量的次序！）。batch_first的默认值式False，此时的输入次序为(sequence length, batch size, number of features, LNF)，如果为True，则是(batch size, sequence length, number of features,NLF)。但是对于隐藏向量，其输入总是(1,N,H)。默认输出为(L,N,H)，如果batch_first为True，则是(N,L,H)。



## <font color = #0288d1>GRUs</font>

### 在前面的RNN网络中，用最后的隐藏状态所包含的信息表示了整个序列的信息。也就是说，我们对序列进行了编码，将其包含的信息转换为了一个隐藏状态。但是，如果前一个隐藏状态包含的信息比新计算的状态更多怎么办？如果数据点添加的信息比之前的隐藏状态更多呢？<font color="purple">这两个问题说明我们只取了最后的隐藏状态，中间输出的隐藏状态的信息并没有很好的利用；以及没有考虑数据信息和隐藏状态信息之间的权重</font>。为此可以改进一下。

### GRU为这两个问题提供了答案。GRU不是简单地计算一个新的隐藏状态，而是尝试对新旧隐藏状态进行加权平均：

### $$ h_{new} = \tanh\left( t_{h} + t_{x} \right)$$
### $$ h' = h_{new}*(1 - z) + h_{old}*z $$

### 也就是说，计算出本次的更新后在和上一次的值进行加权平均后为最终值。通过上面的公式，易知如果z=0，则就是基本的RNN。
### 更进一步，为了进一步平衡隐藏状态和数据之间的贡献，我们在引入一个参数r在激活函数之前对上一个隐藏状态进行加权：

### $$h' = \tanh\left( r*t_{h} + t_{x} \right)*(1 - z) + h*z$$

### <font color="purple">**两个新参数r和z称为门(gate)**</font>，r称为复位门（reset gate）和z称为更新门（update gate）。它们都必须是0到1之间的值，只允许原始值的一小部分通过。
### 每个门产生一个值向量(每个值在0到1之间)，其大小对应于隐藏维度的数量。例如，对于两个隐藏维度，门可能具有\[0.52,0.87\]这样的值。

### 最后的问题是，r和z的值怎么确定？在深度学习中，"某些东西从哪里来"的唯一正确答案是，神经网络。我们将用RNN单元的结构来训练两个门，除了它使用了sigmoid激活函数

![](../pictures/7-31.png)

### 2个RNN单元用来训练r和z。新的候选隐藏状态和原有的隐藏状态经过z参数的加权后得到最后的输出。
### 同样的，PyTorch提供了GRU层nn.GRU。GRU层负责为处理隐藏状态，无论输入序列有多长和RNN层是一样的。它们的参数、输入和输出几乎完全相同，除了一个小的区别：你不能再选择不同的激活函数。

## <font color = #0288d1>LSTM</font>

### 在GRU中<font color="purple">**不能选择激活函数**</font>。尤其是那个双曲正切函数tanh。隐藏状态最好的地方在于它被双曲正切所限制，它保证下一个单元格将在相同的范围内获得隐藏状态。隐藏状态最糟糕的地方在于也在于此，它限制了隐藏状态的取值，以及相应的梯度。
### 既然隐藏状态是有界的，二者不可兼得，那么为什么不能在同一个单元格中使用两个隐藏状态呢？

### <font color="purple">**长短期记忆网络(Long short-term memory，LSTM)**</font>使用两种状态，而不是一种。除了常规的以双曲正切为边界的隐态h外，它还引入了无界的第二状态c。所以，LSTM的最大不同，就是有2个输出。我们首先通过常规的RNN得到一个隐藏状态的候选

### $$g = tanh(t_{hg} + t_{xg})$$

单元输出之一是单元状态，单元状态是上一次的单元状态和本次隐藏状态候选之间的加权平均：

### $$c' = g*i + c*f$$

### 其中i和f是输入门和遗忘门，也是带学习的参数。第二个输出是隐藏状态，是单元输出的双曲正切和输出门的乘积：

### $$h' = \tanh\left( c' \right)*o$$

### 至此，带学习的门共有3个，仍然使用RNN来学习就和GRU一样

![](../pictures/7-32.png)

### PyTorch提供了nn.LSTM对应于LSTM神经网络。LSTM层负责处理隐藏以及单元状态，不管输入序列有多长。